# Reading the dataset

In [8]:
import pandas as pd
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
features_df = pd.read_csv("../data/features_df.csv")
edgelist = pd.read_csv("../data/edgelist.csv")

# Pre-processing

In [ ]:
# save user ids for mapping
user_ids = features_df["user_id"].values

# one-hot encode all categorical
cat_cols = features_df.select_dtypes(include=["object", "bool"]).columns.tolist()
if "user_id" in cat_cols:
    cat_cols.remove("user_id")
df_numeric = features_df.drop(columns=["user_id"])
df_numeric = pd.get_dummies(df_numeric, columns=cat_cols)

# fillna with median
df_numeric = df_numeric.fillna(df_numeric.median())

# Standardize
scaler = StandardScaler()
X = scaler.fit_transform(df_numeric)

# Build Autoencoder

## set model

### first try

In [14]:
input_dim = X.shape[1]
encoding_dim = 32  # hyperparam

input_layer = Input(shape=(input_dim,))

# Encoder
encoded = Dense(64, activation="relu")(input_layer)
encoded = Dense(encoding_dim, activation="relu")(encoded)  # user embedding

# Decoder
decoded = Dense(64, activation="relu")(encoded)
decoded = Dense(input_dim, activation="linear")(decoded)  # restore original dim

In [15]:
autoencoder = Model(inputs=input_layer, outputs=decoded)
encoder = Model(inputs=input_layer, outputs=encoded)  # extract Encoder for embedding

autoencoder.compile(optimizer="adam", loss="mse")

In [16]:
# Train + Eval
# hyperparam
history = autoencoder.fit(
    X, X, epochs=50, batch_size=32, validation_split=0.2, verbose=1
)

# evaluate MSE between decoded & input
mse_loss = autoencoder.evaluate(X, X, verbose=0)
print(f"\nOverall Reconstruction MSE Loss: {mse_loss:.4f}")

Epoch 1/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0703 - val_loss: 0.0200
Epoch 2/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0187 - val_loss: 0.0176
Epoch 3/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0165 - val_loss: 0.0148
Epoch 4/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0128 - val_loss: 0.0103
Epoch 5/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0090 - val_loss: 0.0080
Epoch 6/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0075 - val_loss: 0.0071
Epoch 7/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0067 - val_loss: 0.0065
Epoch 8/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0063 - val_loss: 0.0061
Epoch 9/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0059 - val_loss: 0.0057
Epoch 10/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0055 - val_loss: 0.0054
Epoch 11/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0053 - val_loss: 0.0052
Epoch 12/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0050 - val_lo

Means we lost 15.56% of original info when converted 110+ features to 32 (hopefully
noises we wish to lose).

In [17]:
user_embeddings = encoder.predict(X)

# evaluate cosine similarity
similarity_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(similarity_matrix, index=user_ids, columns=user_ids)

61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [18]:
# Evaluate with edgelist - Recall@10 (if the target in edgelist，appears in autoencoder's Top-10, then it's a hit)
hits = 0
total_edges = 0

for idx, row in edgelist.iterrows():
    anchor = row["user_id_anchor"]
    positive = row["user_id_positive"]

    # make sure both are in feature list
    if anchor in similarity_df.index and positive in similarity_df.index:
        # find anchor's top 10 similar (excluding self)
        top_10_similar = (
            similarity_df.loc[anchor].drop(anchor).nlargest(10).index.tolist()
        )

        if positive in top_10_similar:
            hits += 1
        total_edges += 1

if total_edges > 0:
    hit_rate = hits / total_edges
    print(f"\nMatching Recall@10: {hit_rate:.2%}")
    print(
        f"Out of {total_edges} preset pairs, {hits} were successfully found in the Top 10 matches."
    )
else:
    print("No matching users found between edgelist and features_df.")


Matching Recall@10: 1.63%
Out of 13335 preset pairs, 217 were successfully found in the Top 10 matches.


In [19]:
# --- User-level Recall@10 Evaluation ---

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

unique_anchors = valid_edgelist["user_id_anchor"].unique()

user_hits = 0
total_users = len(unique_anchors)

print(f"Starting User-level Evaluation for {total_users} users...")

for anchor in unique_anchors:
    positives = set(
        valid_edgelist[valid_edgelist["user_id_anchor"] == anchor]["user_id_positive"]
    )

    top_10_similar = set(similarity_df.loc[anchor].drop(anchor).nlargest(10).index)

    # any hit within top 10 counts as a hit for this user
    if len(positives.intersection(top_10_similar)) > 0:
        user_hits += 1

user_recall_rate = user_hits / total_users

print("\n[User-level Results]")
print(f"User-level Recall@10: {user_recall_rate:.2%}")
print(f"Successfully matched {user_hits} out of {total_users} users.")

Starting User-level Evaluation for 1931 users...

[User-level Results]
User-level Recall@10: 10.67%
Successfully matched 206 out of 1931 users.


### second try

In [12]:
input_dim = X.shape[1]
encoding_dim = 64  # larger than first model to retain more info

input_layer = Input(shape=(input_dim,))
# Encoder
x = Dense(128, activation="relu")(input_layer)
x = BatchNormalization()(x)  # added batch normalization
x = Dropout(0.1)(x)  # added dropout
encoded = Dense(encoding_dim, activation="relu")(x)

# Decoder
x = Dense(128, activation="relu")(encoded)
x = BatchNormalization()(x)
decoded = Dense(input_dim, activation="linear")(x)

autoencoder = Model(input_layer, decoded)
encoder = Model(input_layer, encoded)

autoencoder.compile(optimizer="adam", loss="mae")  # changed to MAE might be more r

# increase epochs
autoencoder.fit(X, X, epochs=100, batch_size=64, validation_split=0.1, verbose=0)

# Evaluate
print("Extracting embeddings and calculating similarity...")
user_embeddings = encoder.predict(X)
sim_matrix = cosine_similarity(user_embeddings)

sim_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

# ensure we're getting users in both df
valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

print(f"Testing on {len(valid_edgelist)} pairs...")

# get top 10 match from encoder
hits = 0
for anchor in valid_edgelist["user_id_anchor"].unique():
    # get positives of this anchor in edgelist
    positives = set(
        valid_edgelist[valid_edgelist["user_id_anchor"] == anchor]["user_id_positive"]
    )

    # get this anchor's top 11 similarity (cuz self might be top 1)
    top_n = sim_df.loc[anchor].nlargest(11).index.tolist()
    if anchor in top_n:
        top_n.remove(anchor)
    top_10 = set(top_n[:10])

    # any positive hit would count as one hit
    if len(positives.intersection(top_10)) > 0:
        hits += 1

final_score = hits / len(valid_edgelist["user_id_anchor"].unique())
print(f"New User-level Recall@10: {final_score:.2%}")

Extracting embeddings and calculating similarity...
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Testing on 13335 pairs...
New User-level Recall@10: 11.70%


In [13]:
# Pair level recall：
user_embeddings = encoder.predict(X)
sim_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

print(f"Starting Pair-level Evaluation on {len(valid_edgelist)} pairs...")

hits = 0
total_pairs = len(valid_edgelist)

for idx, row in valid_edgelist.iterrows():
    anchor = row["user_id_anchor"]
    positive = row["user_id_positive"]

    top_10_similar = similarity_df.loc[anchor].drop(anchor).nlargest(10).index.tolist()

    if positive in top_10_similar:
        hits += 1

pair_level_recall = hits / total_pairs

print("\n[Final Results - Pair-level]")
print(
    f"Deep Autoencoder Reconstruction Loss (MAE): {autoencoder.evaluate(X, X, verbose=0):.4f}"
)
print(f"Pair-level Recall@10: {pair_level_recall:.2%}")
print(f"Successfully found {hits} out of {total_pairs} preset pairs.")

61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Starting Pair-level Evaluation on 13335 pairs...

[Final Results - Pair-level]
Deep Autoencoder Reconstruction Loss (MAE): 0.0298
Pair-level Recall@10: 1.90%
Successfully found 253 out of 13335 preset pairs.


### third try

In [9]:
from sklearn.preprocessing import MinMaxScaler  # 换成了 MinMaxScaler

# changed to use min max scaler
scaler = MinMaxScaler()
X = scaler.fit_transform(df_numeric)

# 2. deeper Autoencoder
input_dim = X.shape[1]
encoding_dim = 64  # Bottleneck dim

input_layer = Input(shape=(input_dim,))

# Input -> 256 -> 128 -> 64
# Encoder
encoder_layers = Sequential(
    [
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dense(128, activation="relu"),
        BatchNormalization(),
        Dense(encoding_dim, activation="relu"),  # Latent Space
    ]
)

# Decoder
decoder_layers = Sequential(
    [
        Dense(128, activation="relu"),
        BatchNormalization(),
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dense(
            input_dim, activation="sigmoid"
        ),  # cuz input is now 0-1，so use sigmoid for output
    ]
)

encoded_repr = encoder_layers(input_layer)
decoded_repr = decoder_layers(encoded_repr)

autoencoder = Model(inputs=input_layer, outputs=decoded_repr)
encoder_model = Model(inputs=input_layer, outputs=encoded_repr)

autoencoder.compile(optimizer="adam", loss="mae")  # MAE

# increased epochs
autoencoder.fit(X, X, epochs=100, batch_size=64, validation_split=0.1, verbose=1)

# eval
user_embeddings = encoder_model.predict(X)
sim_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

hits = 0
unique_anchors = valid_edgelist["user_id_anchor"].unique()

for anchor in unique_anchors:
    positives = set(
        valid_edgelist[valid_edgelist["user_id_anchor"] == anchor]["user_id_positive"]
    )

    top_10 = similarity_df.loc[anchor].drop(anchor).nlargest(10).index.tolist()

    if any(p in top_10 for p in positives):
        hits += 1

recall_at_10 = hits / len(unique_anchors)
print("\n[Final Results]")
print(
    f"Deep Autoencoder Reconstruction Loss (MAE): {autoencoder.evaluate(X, X, verbose=0):.4f}"
)
print(f"User-level Recall@10: {recall_at_10:.2%}")
print(f"Matched {hits} users out of {len(unique_anchors)} total anchors in edgelist.")

Epoch 1/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 0.2819 - val_loss: 0.2422
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.1678 - val_loss: 0.1151
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1082 - val_loss: 0.0913
Epoch 4/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0967 - val_loss: 0.0905
Epoch 5/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0885 - val_loss: 0.0837
Epoch 6/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0884 - val_loss: 0.0730
Epoch 7/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0780 - val_loss: 0.0751
Epoch 8/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0755 - val_loss: 0.0700
Epoch 9/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0713 - val_loss: 0.0634
Epoch 10/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0707 - val_loss: 0.0606
Epoch 11/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0656 - val_loss: 0.0574
Epoch 12/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss:

In [11]:
# Pair level recall：
user_embeddings = encoder_model.predict(X)
sim_matrix = cosine_similarity(user_embeddings)
similarity_df = pd.DataFrame(sim_matrix, index=user_ids, columns=user_ids)

valid_edgelist = edgelist[
    edgelist["user_id_anchor"].isin(user_ids)
    & edgelist["user_id_positive"].isin(user_ids)
]

print(f"Starting Pair-level Evaluation on {len(valid_edgelist)} pairs...")

hits = 0
total_pairs = len(valid_edgelist)

for idx, row in valid_edgelist.iterrows():
    anchor = row["user_id_anchor"]
    positive = row["user_id_positive"]

    top_10_similar = similarity_df.loc[anchor].drop(anchor).nlargest(10).index.tolist()

    if positive in top_10_similar:
        hits += 1

pair_level_recall = hits / total_pairs

print("\n[Final Results - Pair-level]")
print(
    f"Deep Autoencoder Reconstruction Loss (MAE): {autoencoder.evaluate(X, X, verbose=0):.4f}"
)
print(f"Pair-level Recall@10: {pair_level_recall:.2%}")
print(f"Successfully found {hits} out of {total_pairs} preset pairs.")

61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Starting Pair-level Evaluation on 13335 pairs...

[Final Results - Pair-level]
Deep Autoencoder Reconstruction Loss (MAE): 0.0265
Pair-level Recall@10: 1.96%
Successfully found 262 out of 13335 preset pairs.
